# CERT r6.2 Feasibility Audit — Cross-Channel Low-and-Slow Data Exfiltration


## 1. Purpose

This audit answers one question:

> **Is CERT r6.2 suitable as the primary dataset for an AI/UEBA project that detects low-and-slow data exfiltration distributed across Email, USB, and HTTP over time?**

The audit was completed **before** rebuilding the data-engineering pipeline so the dataset decision is evidence-based and reproducible.

### Final decision

**GO ✅ — CERT r6.2 is suitable as the primary benign-background dataset for the project.**

The strongest reason is not simply that the three channels exist. The dataset contains a substantial group of benign users who naturally use **all three channels repeatedly and over long overlapping time periods**, which is essential for user-specific baselines and realistic synthetic attack overlays.


## 2. Dataset Snapshot

| Item | Result |
|---|---:|
| Total users with Email Send activity | 4,000 |
| Official malicious users in r6.2 | 5 |
| Approx. benign users available | 3,995 |
| `email.csv` size | 7.530 GB |
| `file.csv` size | 1.240 GB |
| `http.csv` size | 84.036 GB |
| `device.csv` size | 0.130 GB |
| `logon.csv` size | 0.225 GB |

### Relevant schemas

**Email**
- `id, date, user, pc, to, cc, bcc, from, activity, size, attachments, content`
- Activities observed: `View`, `Send`

**File**
- `id, date, user, pc, filename, activity, to_removable_media, from_removable_media, content`
- Activities observed: `File Copy`, `File Open`, `File Delete`, `File Write`

**HTTP**
- `id, date, user, pc, url, activity, content`
- Activities observed: `WWW Visit`, `WWW Download`, `WWW Upload`

This is a major improvement over r4.2 because HTTP now explicitly distinguishes **WWW Upload**, and file activity explicitly includes **File Copy** plus removable-media direction.


## 3. Official Ground Truth

The official `answers/insiders.csv` contains **5 malicious users / 5 scenarios** for r6.2.

| Scenario | User | Start | End | Evidence visible in detailed answer file |
|---:|---|---|---|---|
| 1 | ACM2278 | 2010-08-18 21:47:42 | 2010-08-24 03:48:51 | Logon, removable-device activity, and file activity |
| 2 | CMP2946 | 2011-02-07 12:28:06 | 2011-03-04 12:30:25 | Repeated external job-search emails |
| 3 | PLJ1771 | 2010-08-12 08:09:00 | 2010-08-12 16:11:39 | Email activity plus `keylogger.exe` written |
| 4 | CDE1846 | **2011-02-21 11:43:39** | 2011-04-25 17:55:00 | External personal email with attachments plus file activity |
| 5 | MBG3183 | 2010-10-12 13:21:59 | 2010-10-12 13:22:56 | Four Dropbox `WWW Upload` events |

### Data-quality note

The Scenario 4 `start` value in `insiders.csv` is malformed as `/21/2011 11:43:39`. The detailed `r6.2-4.csv` begins at `02/21/2011 11:43:39`. The corrected value should be documented rather than silently inferred.


## 4. Channel Feasibility

After excluding the five official malicious users:

| Channel behavior | Benign users |
|---|---:|
| True External Email Send | **3,994** |
| File Copy → removable media | **783** |
| HTTP `WWW Upload` | **615** |
| Users with all three behaviors | **319** |

### Why this matters

The project should not inject a USB or HTTP exfiltration action into someone who never uses that channel naturally. The **319-user intersection** gives a realistic population for cross-channel UEBA experiments.


## 5. Active-Day Distributions for the 319 Three-Channel Benign Users

| Metric | External Email Days | USB Copy Days | HTTP Upload Days |
|---|---:|---:|---:|
| Min | 27 | 4 | 12 |
| P10 | 51 | 16 | 69 |
| P25 | 122 | 34 | 80 |
| Median | **232** | **81** | **94** |
| P75 | 271.5 | 145.5 | 116 |
| P90 | 304.2 | 280.2 | 126 |
| P95 | 329.2 | 355 | 139.2 |
| Max | 437 | 486 | 162 |

### Weakest-channel measure

`min_channel_active_days = min(Email days, USB days, HTTP days)`

| Statistic | Minimum active days across all 3 channels |
|---|---:|
| Min | 4 |
| P10 | 16 |
| P25 | 33 |
| Median | **58** |
| P75 | 84 |
| P90 | 108.6 |
| P95 | 121 |
| Max | 162 |

The median three-channel user has at least **58 active days even in the least-used channel**.


## 6. Candidate Availability

| Minimum active days in **each** channel | Eligible users |
|---:|---:|
| ≥ 1 | 319 |
| ≥ 5 | 318 |
| ≥ 10 | 305 |
| ≥ 20 | 274 |
| ≥ 30 | **246** |
| ≥ 50 | **189** |

### Recommended cohorts

**Primary / strict cohort**
- At least **50 active days in each channel**
- At least **180 days common three-channel period**
- **189 users**

**Extended cohort**
- At least **30 active days in each channel**
- At least **180 days common three-channel period**
- **246 users**

The strict cohort is preferable for primary experiments because user-specific statistics such as medians, MAD, activity rates, and high percentiles are more stable with a larger behavioral history.


## 7. Temporal Overlap

| Statistic | Common three-channel period (days) |
|---|---:|
| Min | 22 |
| P10 | 389.4 |
| P25 | 475.5 |
| Median | **498** |
| P75 | 505 |
| P90 | 508 |
| P95 | 509 |
| Max | 515 |

### Final feasibility matrix

| Min active days / channel | ≥30 common days | ≥60 | ≥90 | ≥180 |
|---:|---:|---:|---:|---:|
| 10 | 305 | 304 | 301 | 297 |
| 20 | 274 | 274 | 273 | 272 |
| 30 | **246** | **246** | **246** | **246** |
| 50 | **189** | **189** | **189** | **189** |

The overlap requirement barely reduces the strict cohorts. This confirms that their three-channel activity overlaps over long periods rather than appearing at unrelated times.


## 8. Size / Payload Limitation

### What r6.2 provides

**Email**
- Explicit `Send`
- Message `size`
- Attachment field can contain attachment sizes
- Therefore Email volume is directly observable

**USB**
- Explicit `File Copy`
- Explicit `to_removable_media`
- **No exact transferred byte count**

**HTTP**
- Explicit `WWW Upload`
- URL and content
- **No exact uploaded byte count**

### Scenario 5 recovery test

Dropbox-uploaded files: `46BUD2JL.zip`, `WO83A8RQ.jpg`, `756O1A4U.txt`. Searches found **0 matches** in `file.csv` and Email `attachments`. Therefore no reliable exact byte size could be recovered from another r6.2 source.

### Methodological consequence

Do **not** infer bytes from `content` length. For synthetic attacks, keep exact `payload_bytes` in a separate **attack ground-truth table**. The detector should only receive features observable from the CERT logs.


## 9. Benign vs Malicious: Labeling Logic

An event such as `WWW Upload`, external email, or USB copy is **not automatically malicious**.

### Observable behavior
Examples: External Email Send, File Copy → removable media, WWW Upload.

### Ground truth
The official answer key identifies which original CERT events belong to malicious scenarios.

For synthetic experiments:
1. Start with a benign user's original behavior.
2. Add a controlled low-and-slow attack sequence.
3. Record inserted attack steps in a separate synthetic ground-truth table.
4. Label only injected attack periods/events appropriately.
5. Never relabel untouched benign behavior as malicious.

Suggested fields: `attack_id, user, timestamp, channel, payload_bytes, attack_step, cumulative_payload_bytes, is_synthetic`.


## 10. Final GO / NO-GO Decision

### Decision: **GO ✅**

r6.2 satisfies the critical requirements:

- Natural external Email activity ✅
- Explicit HTTP Upload semantics ✅
- Explicit removable-media file-copy semantics ✅
- Large benign population ✅
- 319 benign users naturally using all three channels ✅
- Repeated activity in each channel ✅
- Long temporal overlap ✅
- Official malicious ground truth ✅
- Enough strict candidates for user-level Train / Validation / Test separation ✅

### Remaining limitation

Exact transferred bytes are not directly available for USB and HTTP. This is acceptable **only if** exact synthetic payload volume is maintained as ground truth and is not leaked into model features.


## 11. Next Phase — Data Engineering

1. Create clean channel event tables.
2. Build a unified `user-day` table.
3. Define a leakage-safe baseline period.
4. Build per-user normality profiles.
5. Engineer temporal and cross-channel behavioral features.
6. Select attack-host users from the strict/extended cohorts.
7. Generate controlled low-and-slow synthetic attacks.
8. Maintain separate synthetic ground truth.
9. Split users into Train / Validation / Test **before** attack generation.
10. Train and evaluate using PR-AUC, Precision, Recall, F1, FPR, and detection delay.

### Key rule

**Do not split the same user's events across train and test.** Use user-level separation so evaluation measures generalization to unseen users rather than memorization.


## 12. Optional Verification Cell

Run this after saving the audit CSVs in `results/`. It verifies the key expected counts.


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\\Users\\HP\\Documents\\FYP\\DATA_ENGINEERING_R62")
RESULTS = PROJECT_ROOT / "results"

channel_frequency = pd.read_csv(RESULTS / "r62_channel_frequency_audit.csv")
temporal_overlap = pd.read_csv(RESULTS / "r62_temporal_overlap_audit.csv")
final_feasibility = pd.read_csv(RESULTS / "r62_final_feasibility_matrix.csv")
primary_candidates = pd.read_csv(RESULTS / "r62_primary_candidate_users.csv")

print("Three-channel users:", len(channel_frequency))
print("Primary candidates:", len(primary_candidates))
print("\nFinal feasibility matrix:")
print(final_feasibility.to_string(index=False))

assert len(channel_frequency) == 319
assert len(primary_candidates) == 189

print("\n✅ SUCCESS — Saved feasibility results verified.")


Three-channel users: 319
Primary candidates: 189

Final feasibility matrix:
 min_active_days_each_channel  min_common_period_days  eligible_users
                           10                      30             305
                           10                      60             304
                           10                      90             301
                           10                     180             297
                           20                      30             274
                           20                      60             274
                           20                      90             273
                           20                     180             272
                           30                      30             246
                           30                      60             246
                           30                      90             246
                           30                     180             246
              